# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

Note: you may need to restart the kernel to use updated packages.

You can now save your data files in: /Users/leonhsu/Projects/GH_AT3/36106/assignment/AT3/data


---
## Student Information

In [2]:
# <Student to fill this section>
group_name = "Group 29"
student_name = "Leon Hsu"
student_id = "26324145"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [6]:
# <Student to fill this section>

### 0.b Import Packages

In [7]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [8]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "account_balance_history.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
# <Student to fill this section>

In [71]:
df.head()

,history_id,account_id,customer_id,balance_date,balance,balance_band,balance_change,stress_indicator
0,10000001,1000001,1000001,2022-01-31,300552.35,Private Banking,2126.80,0
1,10000002,1000001,1000001,2022-02-28,300152.09,Private Banking,-400.26,0
2,10000003,1000001,1000001,2022-03-31,302879.72,Private Banking,2727.64,0
3,10000004,1000001,1000001,2022-04-30,309090.99,Private Banking,6211.27,0
4,10000005,1000001,1000001,2022-05-31,308309.12,Private Banking,-781.87,0


In [72]:
df.shape

(1850976, 8)

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1850976 entries, 0 to 1850975
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   history_id        int64  
 1   account_id        int64  
 2   customer_id       int64  
 3   balance_date      object 
 4   balance           float64
 5   balance_band      object 
 6   balance_change    float64
 7   stress_indicator  int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 113.0+ MB


In [74]:
df.describe()

,history_id,account_id,customer_id,balance,balance_change,stress_indicator
count,1.850976e+06,1.850976e+06,1.850976e+06,1.850976e+06,1.850976e+06,1.850976e+06
mean,1.092549e+07,1.019282e+06,1.007532e+06,4.052007e+05,2.302871e+02,7.815315e+00
std,5.343309e+05,1.113189e+04,4.340480e+03,4.961721e+05,4.936044e+03,9.779524e+00
min,1.000000e+07,1.000001e+06,1.000001e+06,0.000000e+00,-5.049994e+04,0.000000e+00
25%,1.046274e+07,1.009641e+06,1.003754e+06,7.689946e+04,-2.175460e+03,0.000000e+00
50%,1.092549e+07,1.019282e+06,1.007549e+06,2.084024e+05,1.516050e+02,0.000000e+00
75%,1.138823e+07,1.028922e+06,1.011306e+06,4.577770e+05,2.572273e+03,1.000000e+01
max,1.185098e+07,1.038562e+06,1.015000e+06,3.652886e+06,4.576453e+04,9.500000e+01


In [10]:
# <Student to fill this section>
dataset_insights = """
provide a detailed analysis on the dataset, its dimensions, information, issues, ...
"""

In [11]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### Helper Functions

In [12]:
# Chart helpers
# -------------
# Every chart below is built from three small, named pieces instead of one
# big function per chart type:
#   _finish      applies the title/size/axis styling shared by every chart
#   _bar_chart   draws one bar per category — used by every function below
#   _bin_counts  groups a numeric column into readable ranges (e.g. '108–434')
# Each public function (numeric_distribution, categorical_distribution, ...)
# only has to say *what* to plot; the pieces above handle *how* to draw it.
#
# All aggregation happens in pandas before Altair sees the data, which keeps
# charts fast on 1.85M balance records and avoids Altair's 5,000-row embedding limit.

import numpy as np

UNIT = 'Records'          # what one row of `df` represents, used in axis labels
BAR_COLOR = '#2563eb'
AXIS_CFG = dict(domainColor='#9ca3af', tickColor='#9ca3af', labelColor='#000', titleColor='#000')


def _finish(chart, title, width, height):
    """Apply the title, size and axis styling shared by every chart in this notebook."""
    return (chart.properties(title=title, width=width, height=height)
                 .configure_view(strokeWidth=0)
                 .configure_axis(**AXIS_CFG))


def _bar_chart(data, category_field, value_field, *, category_title, value_title,
               sort_order, tooltip, color, horizontal, title, height=None):
    """One bar per category, laid out as columns, or as rows when horizontal=True."""
    if horizontal:
        encoding = dict(
            y=alt.Y(f'{category_field}:N', title=category_title, sort=sort_order, axis=alt.Axis(grid=False)),
            x=alt.X(f'{value_field}:Q', title=value_title, axis=alt.Axis(grid=False)),
        )
        width, height = 520, height or max(250, 18 * len(data))
    else:
        encoding = dict(
            x=alt.X(f'{category_field}:N', title=category_title, sort=sort_order, axis=alt.Axis(labelAngle=0, grid=False)),
            y=alt.Y(f'{value_field}:Q', title=value_title, axis=alt.Axis(grid=False)),
        )
        width, height = max(440, 70 * len(data)), height or 350

    chart = alt.Chart(data).mark_bar(color=color).encode(**encoding, tooltip=tooltip)
    return _finish(chart, title, width, height)


def _decimals_for_width(bin_width):
    """How many decimal places keep a bin label readable, given how wide the bin is."""
    if bin_width >= 5:
        return 0
    if bin_width >= 0.5:
        return 1
    return 2


def _bin_counts(series, bins, min_bins, max_bins):
    """Group a numeric series into readable bins (e.g. '108–434') and count rows per bin."""
    if bins == 'auto':
        # Freedman-Diaconis bin width, clipped to [min_bins, max_bins] so labels stay readable
        edges = np.histogram_bin_edges(series.dropna(), bins='fd')
        bins = int(np.clip(len(edges) - 1, min_bins, max_bins))

    intervals = pd.cut(series, bins=bins, include_lowest=True)
    counts = intervals.value_counts().sort_index().reset_index()
    counts.columns = ['interval', 'Count']

    decimals = _decimals_for_width(counts['interval'].iloc[0].length)
    label = lambda v: f'{v:,.{decimals}f}'
    counts['Bin'] = counts['interval'].apply(lambda iv: f'{label(iv.left)}–{label(iv.right)}')
    return counts[['Bin', 'Count']]


def numeric_distribution(col, bins='auto', discrete=False, min_bins=8, max_bins=20, data=None):
    """Bar chart of how many Records fall into each value (discrete=True) or range of `col`."""
    data = df if data is None else data
    if discrete:
        counts = data[col].value_counts().sort_index().reset_index()
        counts.columns = [col, 'Count']
        plot_data = pd.DataFrame({'Bin': counts[col].astype(str), 'Count': counts['Count']})
    else:
        plot_data = _bin_counts(data[col], bins, min_bins, max_bins)

    return _bar_chart(
        plot_data, 'Bin', 'Count', category_title=col, value_title=f'Number of {UNIT}',
        sort_order=list(plot_data['Bin']), horizontal=False, title=f'Distribution of {col}',
        color=BAR_COLOR, tooltip=[alt.Tooltip('Bin:N', title=col), alt.Tooltip('Count:Q', title=UNIT, format=',')],
    )


def categorical_distribution(col, order=None, top_n=None, horizontal=False, data=None):
    """Bar chart of how many Records fall into each category of `col` (optionally only the top_n)."""
    data = df if data is None else data
    counts = data[col].astype(str).value_counts().reset_index()
    counts.columns = [col, 'Count']
    if top_n is not None:
        counts = counts.head(top_n)
    counts['Share'] = counts['Count'] / len(data)

    sort_order = order if order is not None else ('-x' if horizontal else '-y')
    title = f'Distribution of {col}' if top_n is None else f'Top {top_n} {col} by {UNIT}'
    tooltip = [alt.Tooltip(f'{col}:N'),
               alt.Tooltip('Count:Q', title=UNIT, format=','),
               alt.Tooltip('Share:Q', format='.1%')]

    return _bar_chart(counts, col, 'Count', category_title=col, value_title=f'Number of {UNIT}',
                       sort_order=sort_order, horizontal=horizontal, title=title,
                       color=BAR_COLOR, tooltip=tooltip)

> **Columns not explored individually:** `history_id`, `account_id` and `customer_id` are identifiers. `history_id` is a unique row key, and `account_id` / `customer_id` are pseudonymous keys that link people across tables, so they carry no predictive signal on their own and are only used here for joins and per-account checks. This table has no location fields.

In [13]:
df_cleaned = df.copy()

### B.2 Explore Feature of Interest `balance_date`

> You can add more cells in this section

In [14]:
balance_date_df = df['balance_date']

In [15]:
balance_date_df.describe()

count        1850976
unique            48
top       2022-01-31
freq           38562
Name: balance_date, dtype: object

In [16]:
balance_date_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1850976 entries, 0 to 1850975
Series name: balance_date
Non-Null Count    Dtype 
--------------    ----- 
1850976 non-null  object
dtypes: object(1)
memory usage: 14.1+ MB


In [17]:
balance_date_df.nunique()

48

In [18]:
balance_date_df.isna().sum()

np.int64(0)

In [19]:
# Stored as text (object), so convert to datetime for analysis
balance_date_dt = pd.to_datetime(df['balance_date'], errors='coerce')
print('Unparseable dates:', balance_date_dt.isna().sum())
print('Earliest:', balance_date_dt.min().date(), '| Latest:', balance_date_dt.max().date())
print('Share of month-end dates:', balance_date_dt.dt.is_month_end.mean())
print('Distinct snapshot dates:', balance_date_dt.nunique())

Unparseable dates: 0
Earliest: 2022-01-31 | Latest: 2025-12-31
Share of month-end dates: 1.0
Distinct snapshot dates: 48


In [20]:
# Records per account and duplicate account-month checks
print('Records per account:', df.groupby('account_id').size().value_counts().to_dict())
print('Duplicate account-month records:', df.duplicated(['account_id', 'balance_date']).sum())

Records per account: {48: 38562}
Duplicate account-month records: 0


In [21]:
# Number of accounts reporting a balance each month
monthly_records = balance_date_dt.value_counts().sort_index().reset_index()
monthly_records.columns = ['Month', 'Count']

alt.Chart(monthly_records).mark_line(point=True, color='#2563eb').encode(
    x=alt.X('Month:T', title='Month'),
    y=alt.Y('Count:Q', title='Number of Records', scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip('Month:T', format='%b %Y'), alt.Tooltip('Count:Q', format=',')]
).properties(title='Balance Records per Month', width=620, height=300
).configure_view(strokeWidth=0).configure_axis(**AXIS_CFG)

alt.Chart(...)

In [22]:
# Cross-check against account open dates and status in accounts.csv
accounts_df = pd.read_csv(at.folder_path / "accounts.csv", usecols=['account_id', 'open_date', 'account_status'])
accounts_df['open_date'] = pd.to_datetime(accounts_df['open_date'], errors='coerce')
print('Accounts opened after the first snapshot (2022-01-31):', (accounts_df['open_date'] > '2022-01-31').sum())
print('Accounts opened after the last snapshot (2025-12-31):', (accounts_df['open_date'] > '2025-12-31').sum())
accounts_df['account_status'].value_counts()

Accounts opened after the first snapshot (2022-01-31): 8659
Accounts opened after the last snapshot (2025-12-31): 2957


account_status
Active     34603
Dormant     2758
Closed      1201
Name: count, dtype: int64

In [23]:
# <Student to fill this section>
feature_1_insights = """
`balance_date` is the month-end date of each balance snapshot, it is stored as text (object) and needs converting to
datetime. All values parse cleanly with no missing values, and they cover 48 month-ends from 2022-01-31 to
2025-12-31, where every date falls on the last day of the month.

The table is a perfectly balanced panel: each of the 38,562 accounts has exactly 48 records, one per month, with no
gaps or duplicate account-months, giving 38,562 records every month (1,850,976 in total). This is convenient
for building lag and rolling features, but it is not realistic. In `accounts.csv`, 8,659 accounts (22.5%) were
opened after January 2022, and 2,957 have open dates after December 2025 (some as late as 2029), yet all of them
have balances from January 2022. Likewise, the 1,201 Closed accounts keep reporting balances for every month.

Monthly snapshots also hide intra-month behaviour, such as funds that arrive and leave within days, which is a
key money-laundering pattern, so that behaviour must come from the transactions table. The 2022 data gives a
12-month lookback before transactions begin in 2023, which is useful for baseline features.
"""

In [24]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `balance`

> You can add more cells in this section

In [25]:
balance_df = df['balance']

In [26]:
balance_df.describe()

count    1.850976e+06
mean     4.052007e+05
std      4.961721e+05
min      0.000000e+00
25%      7.689946e+04
50%      2.084024e+05
75%      4.577770e+05
max      3.652886e+06
Name: balance, dtype: float64

In [27]:
balance_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1850976 entries, 0 to 1850975
Series name: balance
Non-Null Count    Dtype  
--------------    -----  
1850976 non-null  float64
dtypes: float64(1)
memory usage: 14.1 MB


In [28]:
balance_df.nunique()

1819919

In [29]:
balance_df.isna().sum()

np.int64(0)

In [30]:
print('Skewness:', round(df['balance'].skew(), 2))
print('Negative balances:', (df['balance'] < 0).sum(), '| Zero balances:', (df['balance'] == 0).sum())
print('Accounts with at least one zero-balance month:', df.loc[df['balance'] == 0, 'account_id'].nunique())
df['balance'].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

Skewness: 1.82
Negative balances: 0 | Zero balances: 2584
Accounts with at least one zero-balance month: 519


0.01    6.805715e+03
0.05    2.205251e+04
0.25    7.689946e+04
0.50    2.084024e+05
0.75    4.577770e+05
0.90    1.214212e+06
0.95    1.548840e+06
0.99    2.070869e+06
Name: balance, dtype: float64

In [31]:
# Bins capped at the 99th percentile so the bulk of the distribution is visible
balance_p99 = df.loc[df['balance'] <= df['balance'].quantile(0.99), ['balance']]
numeric_distribution('balance', data=balance_p99)

alt.Chart(...)

In [32]:
# Mean balance across all accounts over time
balance_trend = (df.assign(month=pd.to_datetime(df['balance_date']))
                   .groupby('month')['balance'].mean().reset_index())

alt.Chart(balance_trend).mark_line(point=True, color='#2563eb').encode(
    x=alt.X('month:T', title='Month'),
    y=alt.Y('balance:Q', title='Mean balance', scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip('month:T', format='%b %Y'), alt.Tooltip('balance:Q', format=',.0f')]
).properties(title='Mean Account Balance by Month', width=620, height=300
).configure_view(strokeWidth=0).configure_axis(**AXIS_CFG)

alt.Chart(...)

In [33]:
# <Student to fill this section>
feature_2_insights = """
`balance` is the account's month-end balance, ranging from 0 to 3,652,885.50 with a mean of 405,200.73, a
median of 208,402.37, a standard deviation of 496,172.09, and no missing values.

The distribution is right-skewed (skewness 1.82), the middle 50% sits between ~76,900 and ~457,800, and the 99th
percentile is ~2.07M. These values are far higher than typical retail banking balances, as the median account
holds over 200k and 43.6% of all records are above 250k, so the customer base looks closer to a private bank than a
retail one. A log transform (log1p, because of the zeros) or robust scaling will be needed.

There are no negative balances, as 2,584 records (across 519 accounts) sit at exactly 0. As `balance_change` shows,
these are months where the balance would have gone negative and was floored at zero, so overdrafts are not
represented. The portfolio is also very stable, as the mean balance moves only from ~399.7k in January 2022 to
~410.7k in December 2025 (+2.7%), with no visible seasonality or response to economic conditions.

For AML, what matters is unusual movement relative to an account's own history (e.g. a sudden build-up then
drain), so relative features such as change versus a rolling average are likely to be more informative than the
absolute balance.
"""

In [34]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `balance_band`

> You can add more cells in this section

In [35]:
balance_band_df = df['balance_band']

In [36]:
balance_band_df.describe()

count             1850976
unique                  5
top       Private Banking
freq               807013
Name: balance_band, dtype: object

In [37]:
balance_band_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1850976 entries, 0 to 1850975
Series name: balance_band
Non-Null Count    Dtype 
--------------    ----- 
1850976 non-null  object
dtypes: object(1)
memory usage: 14.1+ MB


In [38]:
balance_band_df.unique()

array(['Private Banking', 'Premium', 'High', 'Medium', 'Low'],
      dtype=object)

In [39]:
balance_band_df.isna().sum()

np.int64(0)

In [40]:
# Balance range covered by each band
df.groupby('balance_band')['balance'].agg(['count', 'min', 'max', 'median']).sort_values('min')

,count,min,max,median
balance_band,,,,
Low,4170,0.00,999.74,0.000
Medium,25893,1001.64,9999.70,6357.720
High,254722,10000.34,49999.77,32188.035
Premium,759178,50000.25,249999.93,124542.065
Private Banking,807013,250000.19,3652885.50,587697.550


In [41]:
categorical_distribution('balance_band', order=['Low', 'Medium', 'High', 'Premium', 'Private Banking'])

alt.Chart(...)

In [42]:
# How many different bands each account passes through over the 48 months
df.groupby('account_id')['balance_band'].nunique().value_counts().sort_index()

balance_band
1    31435
2     6577
3      507
4       43
Name: count, dtype: int64

In [43]:
# <Student to fill this section>
feature_3_insights = """
`balance_band` is an ordinal grouping of `balance` into five bands, Low (below 1,000), Medium (1,000–10,000),
High (10,000–50,000), Premium (50,000–250,000) and Private Banking (250,000 and above), with no missing values.
Unlike `amount_band` in the transactions table, the band boundaries are applied consistently, as every record
falls in the band its balance implies.

The distribution is an inverted pyramid: Private Banking is the largest band with 807,013 records (43.6%),
followed by Premium (41.0%) and High (13.8%), while Medium (1.4%) and Low (0.23%, 4,170 records) are rare.
A real retail bank would show the opposite shape. The labels are also misleading, since 'High' is only the
third of five bands, below Premium and Private Banking.

Bands are very stable: 31,435 accounts (81.5%) stay in one band for all 48 months, 6,577 move between two bands,
and only 550 pass through three or four. The column is fully derived from `balance`, so it should not be used
alongside it, but a change in band could still be a useful, interpretable event feature. Note that
`accounts.csv` also has a `balance_band` column (the current band), so the two should be named
clearly to avoid confusion when the tables are joined.
"""

In [44]:
# Do not modify this code
print_tile(size="h3", key='feature_3_insights', value=feature_3_insights)

### B.5 Explore Feature of Interest `balance_change`

> You can add more cells in this section

In [45]:
balance_change_df = df['balance_change']

In [46]:
balance_change_df.describe()

count    1.850976e+06
mean     2.302871e+02
std      4.936044e+03
min     -5.049994e+04
25%     -2.175460e+03
50%      1.516050e+02
75%      2.572273e+03
max      4.576453e+04
Name: balance_change, dtype: float64

In [47]:
balance_change_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1850976 entries, 0 to 1850975
Series name: balance_change
Non-Null Count    Dtype  
--------------    -----  
1850976 non-null  float64
dtypes: float64(1)
memory usage: 14.1 MB


In [48]:
balance_change_df.nunique()

1090087

In [49]:
balance_change_df.isna().sum()

np.int64(0)

In [50]:
print('Skewness:', round(df['balance_change'].skew(), 2))
print('Share of positive changes:', round((df['balance_change'] > 0).mean(), 4))
df['balance_change'].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])

Skewness: 0.1
Share of positive changes: 0.519


0.01   -13528.4625
0.05    -7558.1650
0.25    -2175.4600
0.50      151.6050
0.75     2572.2725
0.95     8292.2400
0.99    14471.2475
Name: balance_change, dtype: float64

In [51]:
# Does balance_change equal the month-on-month difference in balance?
balance_sorted = df.sort_values(['account_id', 'balance_date'])
previous_balance = balance_sorted.groupby('account_id')['balance'].shift()
actual_diff = balance_sorted['balance'] - previous_balance
has_previous = previous_balance.notna()
reconciles = np.isclose(actual_diff[has_previous], balance_sorted.loc[has_previous, 'balance_change'], atol=0.02)
print('Records with a previous month:', has_previous.sum())
print('Share that reconcile with the balance difference:', round(reconciles.mean(), 4))
mismatches = balance_sorted.loc[has_previous][~reconciles]
print('Mismatches:', len(mismatches), '| of which balance is 0:', (mismatches['balance'] == 0).sum())

Records with a previous month: 1812414
Share that reconcile with the balance difference: 0.9986
Mismatches: 2584 | of which balance is 0: 2584


In [52]:
# Bins between the 1st and 99th percentiles so the bulk of the distribution is visible
lo, hi = df['balance_change'].quantile([0.01, 0.99])
balance_change_mid = df.loc[df['balance_change'].between(lo, hi), ['balance_change']]
numeric_distribution('balance_change', data=balance_change_mid)

alt.Chart(...)

In [53]:
# Spread of monthly change within each balance band
df.groupby('balance_band')['balance_change'].agg(['mean', 'median', 'std']).round(1)

,mean,median,std
balance_band,,,
High,-59.4,-20.4,2268.8
Low,-1782.5,-1392.7,1940.0
Medium,-268.7,-204.4,1949.9
Premium,133.4,137.8,3519.9
Private Banking,439.3,407.7,6507.9


In [54]:
# <Student to fill this section>
feature_4_insights = """
`balance_change` is the monthly movement in balance, ranging from -50,499.94 to 45,764.53 with a mean of 230.29,
a median of 151.61, a standard deviation of 4,936.04, and no missing values. Unlike `balance`, it is almost
symmetric (skewness 0.10), with 51.9% of months positive and most changes between -13.5k and +14.5k (1st–99th
percentiles), which suggests it was generated as random noise with a small upward drift.

It reconciles with the month-on-month difference in `balance` for 99.86% of records. The 2,584 exceptions are
exactly the zero-balance months, where the recorded change would have pushed the balance below zero and the
balance was floored at 0 instead, so in those months `balance_change` is not the real movement. The first month
(January 2022) also has a change value even though there is no earlier record to compare against.

The size of changes is barely linked to the size of the account (correlation with `balance` r = 0.04): the
standard deviation is ~6.5k in Private Banking but still ~1.9k in the Low band, so small accounts swing by
more than their whole balance and drain to zero, which is why the Low band has a strongly negative mean change
(-1,782). Because this is a net monthly figure, it hides large offsetting inflows and outflows (pass-through
activity), so it should be paired with transaction-level flows, and percentage change needs care because of
zero balances.
"""

In [55]:
# Do not modify this code
print_tile(size="h3", key='feature_4_insights', value=feature_4_insights)

### B.6 Explore Feature of Interest `stress_indicator`

> You can add more cells in this section

In [56]:
stress_indicator_df = df['stress_indicator']

In [57]:
stress_indicator_df.describe()

count    1.850976e+06
mean     7.815315e+00
std      9.779524e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+01
max      9.500000e+01
Name: stress_indicator, dtype: float64

In [58]:
stress_indicator_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1850976 entries, 0 to 1850975
Series name: stress_indicator
Non-Null Count    Dtype
--------------    -----
1850976 non-null  int64
dtypes: int64(1)
memory usage: 14.1 MB


In [59]:
stress_indicator_df.unique()

array([ 0, 15, 20, 10, 35, 85, 25, 50, 60, 65, 45, 40, 95, 70, 90, 75])

In [60]:
stress_indicator_df.isna().sum()

np.int64(0)

In [61]:
# Not a binary flag: count of each distinct value
df['stress_indicator'].value_counts().sort_index()

stress_indicator
0     935826
10    509569
15     31718
20    254770
25     61025
35     38285
40      1487
45     14284
50      2094
60      1103
65       106
70       446
75        79
85       115
90         1
95        68
Name: count, dtype: int64

In [62]:
numeric_distribution('stress_indicator', discrete=True)

alt.Chart(...)

In [63]:
# The indicator splits into a fixed base score plus 50 points in low-balance months
stress_base = df['stress_indicator'] % 50
stress_low_balance = df['stress_indicator'] >= 50
print('Base score constant for every account:', (stress_base.groupby(df['account_id']).nunique() == 1).all())
print('Base score constant for every customer:', (stress_base.groupby(df['customer_id']).nunique() == 1).all())
print('Zero-balance months with the +50 component:', (stress_low_balance & (df['balance'] == 0)).sum(),
      'of', (df['balance'] == 0).sum())
pd.crosstab(stress_low_balance, df['balance'] < 500, rownames=['stress >= 50'], colnames=['balance < 500'])

Base score constant for every account: True
Base score constant for every customer: True
Zero-balance months with the +50 component: 2584 of 2584


balance < 500,False,True
stress >= 50,,
False,1846964,0
True,672,3340


In [64]:
# Median balance and change for each indicator value
df.groupby('stress_indicator')[['balance', 'balance_change']].median().round(0)

,balance,balance_change
stress_indicator,,
0,214060.0,157.0
10,208472.0,164.0
15,178401.0,160.0
20,199244.0,153.0
25,201588.0,153.0
35,185290.0,146.0
40,158928.0,265.0
45,227835.0,133.0
50,0.0,-1446.0


In [65]:
# Mean indicator by year
df.groupby(pd.to_datetime(df['balance_date']).dt.year)['stress_indicator'].mean().round(3)

balance_date
2022    7.752
2023    7.808
2024    7.843
2025    7.859
Name: stress_indicator, dtype: float64

In [66]:
# <Student to fill this section>
feature_5_insights = """
`stress_indicator` is described as a stress flag, but it is not binary. It takes 16 values from 0 to 95 in steps
of 5, with a mean of 7.82 and no missing values; 935,826 records (50.6%) are 0, followed by 10 (27.5%) and 20
(13.8%).

The values combine two components. The first is a base score of 0, 10, 15, 20, 25, 35, 40 or 45 that is fixed for
each customer: it never changes across the 48 months and is the same on all of a customer's accounts. The second
is an extra 50 points added in low-balance months: all 2,584 zero-balance records carry it, and 3,340 of the
4,012 records at 50 or above (83%) have a balance under 500. Apart from this jump, the base score is unrelated to
the account's finances: median balances are similar across base levels, the correlation with `balance` is only
r = -0.03, and the yearly mean barely moves (7.75 in 2022 to 7.86 in 2025).

The column should therefore not be treated as one continuous score. It is better split into a customer-level
stress category (the base) and a low-balance flag (0.22% of records), which is itself derived from `balance` and
would leak information if balance is also used. If stress is a modelling target, the meaningful positive class is
very small and the base component looks pre-assigned rather than driven by behaviour, which limits how much a
model can learn from it.
"""

In [67]:
# Do not modify this code
print_tile(size="h3", key='feature_5_insights', value=feature_5_insights)

## C. Ethical Consideration of Features


In [68]:
# <Student to fill this section and then remove this comment>

In [69]:
# <Student to fill this section and then remove this comment>
ethical_considerations = """
Explain ethical concerns related to the selected features
"""

In [70]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)